<a href="https://colab.research.google.com/github/jyeon6273/DeepNeuralNetworks/blob/main/Assignment_1_CIFAR10_Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIFAR-10 Autoencoder - Assignment 1

## Objectives
- Build a convolutional autoencoder.
- Understand encoder, latent code, decoder, and reconstruction loss.
- Compare latent dimensions.
- Visualize original vs reconstructed images.
- Train a denoising autoencoder.
- Visualize latent features with PCA.


In [2]:
import random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from sklearn.decomposition import PCA

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5
BATCH_SIZE = 128
FAST_MODE = False

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("Device:", DEVICE)

Device: cpu


## 1. Load CIFAR-10

CIFAR-10 has 32×32 RGB images from 10 classes.  
For a normal autoencoder, the target is the input itself:

`input x → encoder → latent z → decoder → reconstruction x'`

The model minimizes reconstruction error between `x` and `x'`.


In [ ]:
transform = transforms.ToTensor()

full_train = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10("./data", train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    full_train, [45000, 5000],
    generator=torch.Generator().manual_seed(SEED)
)

if FAST_MODE:
    train_dataset = Subset(train_dataset, range(12000))
    val_dataset = Subset(val_dataset, range(2000))
    test_dataset = Subset(test_dataset, range(2000))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

classes = full_train.classes
print(classes)


 29%|██▉       | 49.1M/170M [07:46<17:55, 113kB/s]

In [ ]:
images, labels = next(iter(train_loader))
plt.figure(figsize=(12,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(images[i].permute(1,2,0))
    plt.title(classes[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Convolutional Autoencoder

Architecture:

`3×32×32 → 32×16×16 → 64×8×8 → 128×4×4 → latent z → decoder → 3×32×32`

The latent vector is the **bottleneck**.


In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU()
        )

        self.fc_encode = nn.Linear(128*4*4, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 128*4*4)

        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder_conv(x)
        h = h.view(x.size(0), -1)
        return self.fc_encode(h)

    def decode(self, z):
        h = self.fc_decode(z).view(-1, 128, 4, 4)
        return self.decoder_conv(h)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


## 3. Reconstruction Loss

We use Mean Squared Error (MSE):

This is not classification. The model learns to reconstruct the image.


In [ ]:
criterion = nn.MSELoss()

def evaluate(model, loader):
    model.eval()
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(DEVICE)
            recon, _ = model(x)
            loss = criterion(recon, x)
            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)
    return total_loss / total_n

def train_autoencoder(model, epochs=EPOCHS, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}
    start = time.perf_counter()

    for epoch in range(1, epochs+1):
        model.train()
        total_loss, total_n = 0.0, 0

        for x, _ in train_loader:
            x = x.to(DEVICE)
            optimizer.zero_grad()
            recon, _ = model(x)
            loss = criterion(recon, x)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)

        train_loss = total_loss / total_n
        val_loss = evaluate(model, val_loader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(f"Epoch {epoch:02d}/{epochs} | train {train_loss:.5f} | val {val_loss:.5f}")

    return history, time.perf_counter() - start


# Task 1 — Basic Autoencoder

Start with `latent_dim = 64`.


In [ ]:
set_seed()
model64 = ConvAutoencoder(latent_dim=64)
history64, train_time = train_autoencoder(model64, epochs=EPOCHS)
print("Test loss:", evaluate(model64, test_loader))
print("Train time:", round(train_time,1), "s")


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history64["train_loss"], marker="o", label="Train")
plt.plot(history64["val_loss"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE reconstruction loss")
plt.title("CIFAR-10 Autoencoder Loss")
plt.grid(alpha=.25)
plt.legend()
plt.show()


In [ ]:
def show_reconstructions(model, loader, n=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(DEVICE)
    with torch.no_grad():
        recon, _ = model(x)

    plt.figure(figsize=(14,4))
    for i in range(n):
        plt.subplot(2,n,i+1)
        plt.imshow(x[i].cpu().permute(1,2,0))
        plt.axis("off")
        if i == 0: plt.ylabel("Original")

        plt.subplot(2,n,n+i+1)
        plt.imshow(recon[i].cpu().permute(1,2,0).clamp(0,1))
        plt.axis("off")
        if i == 0: plt.ylabel("Reconstructed")
    plt.tight_layout()
    plt.show()

show_reconstructions(model64, test_loader)


# Task 2 — Latent Dimension

Compare `32, 64, 128, 256`.

Expected idea:
- smaller latent space → stronger compression, more information loss
- larger latent space → easier reconstruction, less compression


In [ ]:
latent_dims = [32, 64, 128, 256]
latent_models = {}
latent_histories = {}
results = []

for d in latent_dims:
    print("\nLATENT DIM =", d)
    set_seed()
    model = ConvAutoencoder(d)
    history, t = train_autoencoder(model, epochs=EPOCHS)
    test_loss = evaluate(model, test_loader)

    latent_models[d] = model
    latent_histories[d] = history
    results.append({
        "Latent Dimension": d,
        "Test Reconstruction Loss": test_loss,
        "Train Time (s)": t
    })

pd.DataFrame(results)


In [ ]:
plt.figure(figsize=(8,5))
for d in latent_dims:
    plt.plot(latent_histories[d]["val_loss"], marker="o", label=f"z={d}")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Effect of Latent Dimension")
plt.grid(alpha=.25)
plt.legend()
plt.show()


In [ ]:
x, _ = next(iter(test_loader))
x = x[:6].to(DEVICE)

plt.figure(figsize=(12,10))
for i in range(6):
    plt.subplot(len(latent_dims)+1, 6, i+1)
    plt.imshow(x[i].cpu().permute(1,2,0))
    plt.axis("off")
    if i == 0: plt.ylabel("Original")

for row, d in enumerate(latent_dims, start=1):
    latent_models[d].eval()
    with torch.no_grad():
        recon, _ = latent_models[d](x)
    for i in range(6):
        plt.subplot(len(latent_dims)+1, 6, row*6+i+1)
        plt.imshow(recon[i].cpu().permute(1,2,0).clamp(0,1))
        plt.axis("off")
        if i == 0: plt.ylabel(f"z={d}")

plt.tight_layout()
plt.show()


# Task 3 — Denoising Autoencoder

Input = noisy image  
Target = clean image


In [ ]:
def add_noise(x, std=0.20):
    return torch.clamp(x + std*torch.randn_like(x), 0, 1)

def train_denoising(model, noise_std=0.20, epochs=EPOCHS):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(1, epochs+1):
        model.train()
        total_loss, total_n = 0.0, 0

        for clean, _ in train_loader:
            clean = clean.to(DEVICE)
            noisy = add_noise(clean, noise_std)

            optimizer.zero_grad()
            recon, _ = model(noisy)
            loss = criterion(recon, clean)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()*clean.size(0)
            total_n += clean.size(0)

        print(f"Epoch {epoch:02d}/{epochs} | loss {total_loss/total_n:.5f}")

    return model

set_seed()
denoise_model = ConvAutoencoder(64)
denoise_model = train_denoising(denoise_model, noise_std=0.20)


In [ ]:
clean, _ = next(iter(test_loader))
clean = clean[:8].to(DEVICE)
noisy = add_noise(clean, 0.20)

denoise_model.eval()
with torch.no_grad():
    recon, _ = denoise_model(noisy)

plt.figure(figsize=(14,6))
for i in range(8):
    plt.subplot(3,8,i+1)
    plt.imshow(clean[i].cpu().permute(1,2,0)); plt.axis("off")
    if i == 0: plt.ylabel("Clean")

    plt.subplot(3,8,8+i+1)
    plt.imshow(noisy[i].cpu().permute(1,2,0)); plt.axis("off")
    if i == 0: plt.ylabel("Noisy")

    plt.subplot(3,8,16+i+1)
    plt.imshow(recon[i].cpu().permute(1,2,0).clamp(0,1)); plt.axis("off")
    if i == 0: plt.ylabel("Denoised")

plt.tight_layout()
plt.show()


# Experiment 4 — Latent Space Visualization

The autoencoder uses a 64-D latent space.  
For visualization only, we project it to 2D with PCA.

Do not expect perfect class separation because the autoencoder was trained for reconstruction, not classification.


In [ ]:
def collect_latents(model, loader, max_samples=5000):
    model.eval()
    Z, Y = [], []
    count = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            z = model.encode(x).cpu()
            Z.append(z)
            Y.append(y)
            count += x.size(0)
            if count >= max_samples:
                break

    return torch.cat(Z)[:max_samples].numpy(), torch.cat(Y)[:max_samples].numpy()

z, y = collect_latents(model64, test_loader)
z2 = PCA(n_components=2).fit_transform(z)

plt.figure(figsize=(9,7))
for c, name in enumerate(classes):
    mask = y == c
    plt.scatter(z2[mask,0], z2[mask,1], s=10, alpha=.55, label=name)

plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("CIFAR-10 Latent Space: 64D → PCA 2D")
plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.show()
